# 16.4 — BCG predict → eval → paper-style figures

Run this after Josh hands over the **v08-retrained** scANVI control / treated files.
Kernel: **`analysis`**. Predict/eval switch envs themselves; figures stay here.

Do **not** log1p the mouse files first. They should be human ENSG with atlas-posed
`layers['counts']` (continuous, count-scale). This notebook:

1. Checks those files look like that (stop here if they do not)
2. Runs `predict_new_input.sh --posed-ensg` for the chosen `--model-set`
3. Checks the aligned + predicted outputs
4. Puts the human unvaccinated ground truth on the same gene axis
5. Scores control-mouse predictions vs real unvaccinated human
   (`scripts/eval_external_target.py`)
6. Writes the paper-style boards (mean-R², decoded MMD, HSC KDEs, UMAP, means)
   via `scripts/plot_bcg_paper_figures.py`

The same eval+figure step is also a CLI wrapper, so a later atlas model or BCG
correction does not need new plot code:

```bash
bash scripts/score_bcg_and_plot.sh \
  --model-set uncapped_v08_iid --flavor pearson_residuals \
  --tag bcg_ctrl_a2 --target $HUMAN_OUT07
```

Change **`MODEL_SET` / `FLAVOR` / tags / input paths** in the next cell. That is
the whole variant surface. Do not recode the figures.

`RUN_PREDICT` / `RUN_EVAL` / `RUN_PLOT` start as **False**. Verify first, then flip.

**Section 9** is the same pipeline in one cell (verify → predict → human → eval →
figures). Use the stepwise cells to debug; use section 9 for start-to-end.


In [ ]:
import os
import subprocess
import sys

import h5py
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy.sparse as sp_sparse

REPO = "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT"
D = os.path.join(REPO, "cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg")
PREDICT_SH = os.path.join(REPO, "scripts/predict_new_input.sh")
H5AD_TO_V07 = os.path.join(REPO, "scripts/h5ad_to_v07.py")
EVAL_PY = os.path.join(REPO, "scripts/eval_external_target.py")
PLOT_PY = os.path.join(REPO, "scripts/plot_bcg_paper_figures.py")
SCORE_SH = os.path.join(REPO, "scripts/score_bcg_and_plot.sh")
CELLOT_PY = os.path.expanduser("~/.conda/envs/CellOT/bin/python")
ANALYSIS_PY = sys.executable  # this notebook is the analysis env

# Same table as predict_new_input.sh / score_bcg_and_plot.sh.
# Flip MODEL_SET / FLAVOR for a later atlas model; do not recode figures.
MODEL_SETS = {
    "atlas_full_v07": {
        "results": "atlas_full_{flavor}",
        "axis": "hvg_{flavor}_atlas_full_v07.h5ad",
        "suffix": "",
        "short": "v07",
    },
    "uncapped_v08": {
        "results": "hvg_{flavor}_uncapped_v08",
        "axis": "hvg_{flavor}_a_uncapped_v08.h5ad",
        "suffix": "_uncapped_v08",
        "short": "v08",
    },
    "uncapped_v08_iid": {
        "results": "hvg_{flavor}_a_uncapped_v08_iid",
        "axis": "hvg_{flavor}_a_uncapped_v08.h5ad",
        "suffix": "_uncapped_v08_iid",
        "short": "v08iid",
    },
}

MODEL_SET = "uncapped_v08_iid"
FLAVOR = "pearson_residuals"  # or mixhvg once you want that axis
ms = MODEL_SETS[MODEL_SET]
FLAV = f"{FLAVOR}{ms['suffix']}"
AXIS_H5AD = os.path.join(D, ms["axis"].format(flavor=FLAVOR))
GENE_TXT = os.path.join(D, f"gene_lists/hvg_{FLAVOR}_a_uncapped_v08_genes.txt")
GENE_CSV = os.path.join(D, f"gene_lists/hvg_{FLAVOR}_a_uncapped_v08_genes.csv")
AEDIR = os.path.join(REPO, "cellot/cellot_gpu/results",
                     ms["results"].format(flavor=FLAVOR), "scgen")
EVAL_ROOT = os.path.join(REPO, "results/external_eval")

# --- fill these when the v08 scANVI files arrive ---
MOUSE_CTRL = "/path/to/bcg_control_scanvi_v08.h5ad"   # unvaccinated / day-0
MOUSE_TRT = "/path/to/bcg_treated_scanvi_v08.h5ad"    # vaccinated / day-28
HUMAN_CTRL = "/path/to/bcg_human_unvax.h5ad"          # raw integer counts, ENSG names

TAG_CTRL = "bcg_ctrl_a2"
TAG_TRT = "bcg_trt_a2"
EVAL_IMPACT_TAG = f"{TAG_CTRL}_impact_{FLAVOR}_{ms['short']}"
EVAL_SCGEN_TAG = f"{TAG_CTRL}_scgen_{FLAVOR}_{ms['short']}"
PLOT_LABEL = f"BCG {TAG_CTRL} · {FLAVOR} · {MODEL_SET}"
PLOT_OUT = os.path.join(
    REPO, "speciesOT/baseline/analysis/paper_style_bcg_outputs",
    f"{TAG_CTRL}_{FLAVOR}_{MODEL_SET}",
)

HUMAN_OUT = os.path.join(D, f"bcg_human_unvax_target_{FLAV}.h5ad")
HUMAN_OUT07 = os.path.join(D, f"bcg_human_unvax_target_{FLAV}_anndata07.h5ad")

RUN_PREDICT = False
RUN_EVAL = False
RUN_PLOT = False


def _strip(g):
    g = str(g)
    return g.split(".")[0] if g.startswith("ENS") and "." in g else g


def load_axis_genes():
    if os.path.exists(GENE_TXT):
        return [g.strip() for g in open(GENE_TXT) if g.strip()]
    a = ad.read_h5ad(AXIS_H5AD, backed="r")
    genes = [_strip(g) for g in a.var_names.astype(str)]
    a.file.close()
    return genes


def aligned_paths(tag):
    return {
        "aligned": os.path.join(D, f"{tag}_aligned_{FLAV}.h5ad"),
        "aligned07": os.path.join(D, f"{tag}_aligned_{FLAV}_anndata07.h5ad"),
        "pred_impact": os.path.join(D, f"{tag}_predicted_human_via_impact_cellot_{FLAV}.h5ad"),
        "pred_scgen": os.path.join(D, f"{tag}_predicted_human_via_scgen_{FLAV}.h5ad"),
    }


axis_genes = load_axis_genes()
assert len(axis_genes) == 1000, len(axis_genes)
print(f"axis {FLAVOR} {MODEL_SET}: {len(axis_genes)} genes")
print(f"  gene list: {GENE_TXT if os.path.exists(GENE_TXT) else AXIS_H5AD}")
print(f"  aedir: {AEDIR}")
print(f"  eval tags: {EVAL_IMPACT_TAG}  |  {EVAL_SCGEN_TAG}")
print(f"  figures: {PLOT_OUT}")
for label, p in [("MOUSE_CTRL", MOUSE_CTRL), ("MOUSE_TRT", MOUSE_TRT),
                 ("HUMAN_CTRL", HUMAN_CTRL), ("predict.sh", PREDICT_SH),
                 ("plot.py", PLOT_PY), ("score.sh", SCORE_SH), ("aedir", AEDIR)]:
    print(("OK " if os.path.exists(p) else "MISSING ") + f"{label}: {p}")


## 1. Verify the batch-corrected mouse files

What we expect for `--posed-ensg`:

| Check | Pass |
|---|---|
| `.var_names` | human `ENSG…` |
| `layers['counts']` | present, **continuous** (not integers), count-scale (max typically ≫ 15) |
| `layers['counts_original']` | optional; if present, **integers** (raw UMIs) |
| row sums of posed vs original | roughly the same per cell (library-size rescale) |
| overlap with v08 Pearson | Josh saw **996 / 1000**; Aug-5 files were only 498 / 1000 |
| `.X` | ignore (often leftover log-like) |

If `layers['counts']` is already integer, this is probably `counts_original` (attempt 1),
not the posed decode. If max(`counts`) < 15, someone may have already log1p'd — **do not**
send that through the script.


In [ ]:
def _dense(X):
    return X.toarray() if sp_sparse.issparse(X) else np.asarray(X)


def _strip(g):
    g = str(g)
    return g.split(".")[0] if g.startswith("ENS") and "." in g else g


def verify_posed_mouse(path, label, axis, expect_min_overlap=900):
    print("=" * 72)
    print(label, path)
    if not os.path.exists(path) or path.startswith("/path/to/"):
        print("  SKIP: path not set or file missing")
        return None
    a = sc.read_h5ad(path)
    names = [_strip(g) for g in a.var_names.astype(str)]
    n_ensg = sum(g.startswith("ENSG") for g in names)
    layers = list(a.layers.keys()) if a.layers else []
    print(f"  shape {a.shape}  layers {layers}")
    print(f"  obs sample {list(a.obs.columns)[:12]}")
    print(f"  var sample {names[:5]}")
    print(f"  ENSG in var_names: {n_ensg}/{a.n_vars}")
    if "cell_type" in a.obs:
        print("  cell_type", a.obs["cell_type"].astype(str).value_counts().to_dict())

    checks = []
    checks.append(("human ENSG names", n_ensg >= max(10, int(0.8 * a.n_vars))))
    checks.append(("layers has counts", "counts" in a.layers))

    if "counts" in a.layers:
        C = _dense(a.layers["counts"]).astype(np.float64)
        frac_int = float(np.mean(np.isclose(C, np.round(C), atol=1e-5)))
        print(f"  layers['counts']: min={C.min():.4g} max={C.max():.4g} mean={C.mean():.4g}  frac_integer={frac_int:.4f}")
        checks.append(("counts are continuous (posed, not UMIs)", frac_int < 0.05))
        checks.append(("counts look count-scale, not already log1p", float(C.max()) > 15))
        if "counts_original" in a.layers:
            O = _dense(a.layers["counts_original"]).astype(np.float64)
            o_int = float(np.mean(np.isclose(O, np.round(O), atol=1e-5)))
            lib_c = C.sum(axis=1)
            lib_o = O.sum(axis=1)
            rel = np.abs(lib_c - lib_o) / np.maximum(lib_o, 1e-6)
            print(f"  layers['counts_original']: max={O.max():.4g} frac_integer={o_int:.4f}")
            print(f"  per-cell library |posed-orig|/orig: median={np.median(rel):.3g} max={rel.max():.3g}")
            checks.append(("counts_original are integers", o_int > 0.99))
            checks.append(("posed library ≈ original UMI total", float(np.median(rel)) < 0.05))
    else:
        checks.append(("layers has counts", False))

    X = _dense(a.X)
    print(f"  .X (ignore): min={X.min():.4g} max={X.max():.4g} mean={X.mean():.4g}")

    overlap = len(set(names) & set(axis))
    print(f"  overlap with {FLAVOR} {MODEL_SET}: {overlap} / {len(axis)}")
    checks.append((f"axis overlap ≥ {expect_min_overlap}", overlap >= expect_min_overlap))
    if overlap < 700:
        print("  WARNING: this looks like the old v07/685-gene export, not the v08 retrain.")

    print("\n  checklist:")
    ok = True
    for name, passed in checks:
        print(("    PASS  " if passed else "    FAIL  ") + name)
        ok = ok and passed
    print("  overall:", "READY for --posed-ensg" if ok else "NOT READY — fix with Josh before predict")
    return {"adata": a, "overlap": overlap, "ok": ok, "names": names}


In [ ]:
ctrl_info = verify_posed_mouse(MOUSE_CTRL, "control / unvaccinated", axis_genes)
trt_info = verify_posed_mouse(MOUSE_TRT, "treated / vaccinated", axis_genes)


## 2. Predict human cells

Same command the mentor can run in a shell. Flip `RUN_PREDICT = True` in the paths cell
after section 1 is all PASS. `--model-set` comes from the paths cell.

The script still does `normalize_total` + `log1p` once. `--posed-ensg` skips the
integer check and the mouse→human hop. It writes **both** flavors in the set
(pearson + mixhvg for v08); later cells score the `FLAVOR` you chose.


In [ ]:
def predict_cmd(mouse_path, tag):
    return [
        "bash", PREDICT_SH,
        "--model-set", MODEL_SET,
        "--posed-ensg",
        mouse_path,
        tag,
    ]


for mouse_path, tag in [(MOUSE_CTRL, TAG_CTRL), (MOUSE_TRT, TAG_TRT)]:
    cmd = predict_cmd(mouse_path, tag)
    print("\n$", " ".join(cmd))
    if not RUN_PREDICT:
        print("  (dry run — set RUN_PREDICT = True to execute)")
        continue
    if not os.path.exists(mouse_path) or mouse_path.startswith("/path/to/"):
        raise FileNotFoundError(mouse_path)
    proc = subprocess.run(cmd, cwd=REPO, check=False)
    print("  exit", proc.returncode)
    if proc.returncode != 0:
        raise RuntimeError(f"predict_new_input.sh failed for {tag}")


## 3. Verify aligned + predicted outputs

After a successful run, Phase 1 prints `coverage N/1000`. Confirm here that the
aligned matrix is **exactly** this model-set's gene order (the model will refuse
otherwise).


In [ ]:
def verify_aligned(path, axis, label):
    print("=" * 72)
    print(label, path)
    if not os.path.exists(path):
        print("  MISSING (run section 2 with RUN_PREDICT = True)")
        return None
    a = sc.read_h5ad(path)
    names = [_strip(g) for g in a.var_names.astype(str)]
    X = _dense(a.X)
    print(f"  shape {a.shape}")
    print(f"  .X min={X.min():.4g} max={X.max():.4g} mean={X.mean():.4g}")
    print(f"  n_genes=={len(axis)}: {a.n_vars == len(axis)}")
    print(f"  gene order == axis list: {names == list(axis)}")
    if names != list(axis):
        print("  first mismatch:", next(
            ((i, names[i], axis[i]) for i in range(min(len(names), len(axis))) if names[i] != axis[i]),
            None,
        ))
    # log1p(CP10k) typically lives in ~0–8; posed raw counts were tens–hundreds
    print(f"  looks log1p-scale (max < 15): {float(X.max()) < 15}")
    return a


for tag in (TAG_CTRL, TAG_TRT):
    paths = aligned_paths(tag)
    print(f"\n### {tag} files on disk  (flavor={FLAV})")
    for k, p in paths.items():
        print(("  OK " if os.path.exists(p) else "  -- ") + f"{k}: {p}")
    verify_aligned(paths["aligned"], axis_genes, f"{tag} aligned")
    if os.path.exists(paths["pred_impact"]):
        verify_aligned(paths["pred_impact"], axis_genes, f"{tag} IMPACT pred")


## 4. Human unvaccinated ground truth → same gene axis

Human file must be **raw integer counts** with ENSG in `.var_names`.
`predict_new_input.sh` is mouse-only, so this cell does the projection + Scanpy
onto the `FLAVOR` / `MODEL_SET` axis from the paths cell.


In [ ]:
if not os.path.exists(HUMAN_CTRL) or HUMAN_CTRL.startswith("/path/to/"):
    print("HUMAN_CTRL not set — skip. Fill the path and re-run this cell.")
else:
    src = sc.read_h5ad(HUMAN_CTRL)
    if "counts" in src.layers:
        src.X = src.layers["counts"].astype(np.float32)
    X = _dense(src.X).astype(np.float32)
    xs = X[:50].ravel()
    print("human source", src.shape, "var sample", list(src.var_names[:5]))
    print("integer counts?", bool(np.allclose(xs, np.round(xs))))
    if not np.allclose(xs, np.round(xs)):
        raise ValueError("human target must be raw integer counts")
    pos = {_strip(g): i for i, g in enumerate(src.var_names.astype(str))}
    Xn = np.zeros((src.n_obs, len(axis_genes)), dtype=np.float32)
    hit = 0
    for j, g in enumerate(axis_genes):
        i = pos.get(_strip(g))
        if i is not None:
            Xn[:, j] = X[:, i]
            hit += 1
    a = ad.AnnData(
        X=Xn,
        obs=src.obs.copy(),
        var=pd.DataFrame(index=pd.Index(axis_genes, name="ensg")),
    )
    sc.pp.normalize_total(a, target_sum=1e4)
    sc.pp.log1p(a)
    a.write_h5ad(HUMAN_OUT)
    print(f"human coverage {hit}/{len(axis_genes)} ({100*hit/len(axis_genes):.1f}%) -> {HUMAN_OUT}")
    if hit < 900:
        print("WARNING: human coverage is low — scoring will be dominated by zeros.")


## 5. Rewrite human target for the CellOT env (anndata 0.7)

The `_anndata07` suffix is the **file format**, not model v07.


In [ ]:
if not os.path.exists(HUMAN_OUT):
    print("HUMAN_OUT missing — run section 4 first")
else:
    cmd = [CELLOT_PY, H5AD_TO_V07, HUMAN_OUT, HUMAN_OUT07]
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)
    print("wrote", HUMAN_OUT07)


## 6. Score control-mouse predictions vs real unvaccinated human

`--aedir` is this model-set's **scgen** folder even when the prediction is IMPACT
(the OT map has no decoder of its own).

Read **`model_over_floor`** and **`r2_model_dec`**. Do not rank BCG on
`frac_gap_closed_decoded` (small, model-dependent denominator).

Only score treated-mouse preds against a vaccinated human file (repeat 4–5 on that
object). Do not score treated mouse vs unvaccinated human if the question is the
species map.

The CLI equivalent of sections 6–8 is `scripts/score_bcg_and_plot.sh`.


In [ ]:
def eval_cmd(pred, tag):
    src = aligned_paths(TAG_CTRL)["aligned07"]
    return [
        CELLOT_PY, EVAL_PY,
        "--pred", pred,
        "--target", HUMAN_OUT07,
        "--source", src,
        "--aedir", AEDIR,
        "--tag", tag,
    ]


jobs = [
    (aligned_paths(TAG_CTRL)["pred_impact"], EVAL_IMPACT_TAG),
    (aligned_paths(TAG_CTRL)["pred_scgen"], EVAL_SCGEN_TAG),
]
for pred, tag in jobs:
    cmd = eval_cmd(pred, tag)
    print("\n$", " ".join(cmd))
    if not RUN_EVAL:
        print("  (dry run — set RUN_EVAL = True to execute)")
        continue
    for p in (pred, HUMAN_OUT07, aligned_paths(TAG_CTRL)["aligned07"]):
        if not os.path.exists(p):
            raise FileNotFoundError(p)
    subprocess.run(cmd, cwd=REPO, check=True)


## 7. Read the scorecards


In [ ]:
keep = [
    "tag", "ncells", "model_over_floor", "r2_model_dec",
    "mmd_model", "mmd_ae_recon_floor", "mmd_decoded_ceiling",
    "frac_gap_closed_decoded", "mean_js",
]
rows = []
for tag in (EVAL_IMPACT_TAG, EVAL_SCGEN_TAG):
    p = os.path.join(EVAL_ROOT, tag, "external_target_metrics.csv")
    print(("OK " if os.path.exists(p) else "MISSING ") + p)
    if os.path.exists(p):
        df = pd.read_csv(p)
        df["tag"] = tag
        rows.append(df)
if rows:
    out = pd.concat(rows, ignore_index=True)
    cols = [c for c in keep if c in out.columns]
    print(out[cols].to_string(index=False))
    print("\nTrust model_over_floor and r2_model_dec. Print frac_gap_closed_decoded;")
    print("do not rank BCG variants on it (noisy, model-dependent denominator).")


## 8. Paper-style BCG figures

These are the same five boards as the atlas paper-style set, rewritten for BCG
and wired so a later atlas model or BCG correction does not need new plot code.

| file | what it answers | BCG rule |
|---|---|---|
| `scatter_mean_r2` | did the mean land? | pred vs **human BCG**; red = HSC genes on this axis; R²_top100 = Wilcoxon human vs mouse BCG |
| `mmd_bars` | did the cloud land? | **decoded-frame** floor / ceiling / model from the eval CSV — matches the scorecard |
| `kde_markers` | per-gene shape | labels are mouse BCG / human BCG / IMPACT / scGen. scANVI is smoother than raw UMIs |
| `umap_joint` | cloud overlap | grey = observed human; Identity = mouse (no transport). ~400 cells look thin |
| `dotplot_markers` | mean panel only | size encoding **off** — after scANVI almost no exact zeros, so % expressed saturates |

Output: `speciesOT/baseline/analysis/paper_style_bcg_outputs/<tag>_<flavor>_<model_set>/`

CLI (after sections 4–5 exist on disk):

```bash
bash scripts/score_bcg_and_plot.sh \
  --model-set $MODEL_SET --flavor $FLAVOR \
  --tag $TAG_CTRL --target $HUMAN_OUT07
```

`--skip-eval` redraws figures from CSVs already written by section 6.
Flip `RUN_PLOT = True` in the paths cell.

In [ ]:
from IPython.display import Image, display

def plot_cmd():
    src = aligned_paths(TAG_CTRL)
    cmd = [
        ANALYSIS_PY, PLOT_PY,
        "--source", src["aligned"] if os.path.exists(src["aligned"]) else src["aligned07"],
        "--target", HUMAN_OUT if os.path.exists(HUMAN_OUT) else HUMAN_OUT07,
        "--pred-impact", src["pred_impact"],
        "--pred-scgen", src["pred_scgen"],
        "--eval-impact", os.path.join(EVAL_ROOT, EVAL_IMPACT_TAG, "external_target_metrics.csv"),
        "--eval-scgen", os.path.join(EVAL_ROOT, EVAL_SCGEN_TAG, "external_target_metrics.csv"),
        "--outdir", PLOT_OUT,
        "--label", PLOT_LABEL,
        "--source-label", "mouse BCG",
        "--target-label", "human BCG",
    ]
    if os.path.exists(GENE_CSV):
        cmd += ["--symbols-csv", GENE_CSV]
    return cmd


cmd = plot_cmd()
print("$", " ".join(cmd))
if not RUN_PLOT:
    print("  (dry run — set RUN_PLOT = True to execute)")
else:
    for p in cmd[2:]:
        if p.startswith("--"):
            continue
        if p in (PLOT_OUT, PLOT_LABEL, "mouse BCG", "human BCG"):
            continue
        if not os.path.exists(p):
            raise FileNotFoundError(p)
    subprocess.run(cmd, cwd=REPO, check=True)

stats_p = os.path.join(PLOT_OUT, "figure_stats.json")
if os.path.exists(stats_p):
    print(open(stats_p).read())
for stem in ("scatter_mean_r2", "mmd_bars", "kde_markers", "umap_joint", "dotplot_markers"):
    png = os.path.join(PLOT_OUT, f"{stem}.png")
    if os.path.exists(png):
        print(png)
        display(Image(filename=png, width=720))
    elif RUN_PLOT:
        print("MISSING", png)


## 9. Start-to-end (one cell)

Self-contained. Fill the three paths, then run this cell. It stops if the
posed-file checklist fails. Treated mouse is predicted but only **control** is
scored against unvaccinated human. After eval it writes the paper-style boards
(same scripts as sections 6–8).

`MODEL_SET` / `FLAVOR` default to the v08 IID Pearson pair. Change them here if
this run is a different atlas model.


In [ ]:
# === fill these three paths, then run this cell ===
MOUSE_CTRL = "/path/to/bcg_control_scanvi_v08.h5ad"   # unvaccinated / day-0
MOUSE_TRT = "/path/to/bcg_treated_scanvi_v08.h5ad"    # vaccinated / day-28
HUMAN_CTRL = "/path/to/bcg_human_unvax.h5ad"          # raw integer counts, ENSG names

# Flip these for a later atlas model; figures follow automatically.
MODEL_SET = "uncapped_v08_iid"
FLAVOR = "pearson_residuals"
TAG_CTRL, TAG_TRT = "bcg_ctrl_a2", "bcg_trt_a2"

import os, subprocess, sys
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy.sparse as sp_sparse
from IPython.display import Image, display

REPO = "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT"
D = os.path.join(REPO, "cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg")
PREDICT_SH = os.path.join(REPO, "scripts/predict_new_input.sh")
H5AD_TO_V07 = os.path.join(REPO, "scripts/h5ad_to_v07.py")
SCORE_SH = os.path.join(REPO, "scripts/score_bcg_and_plot.sh")
CELLOT_PY = os.path.expanduser("~/.conda/envs/CellOT/bin/python")
EVAL_ROOT = os.path.join(REPO, "results/external_eval")

MODEL_SETS = {
    "atlas_full_v07": {"results": "atlas_full_{flavor}", "axis": "hvg_{flavor}_atlas_full_v07.h5ad",
                      "suffix": "", "short": "v07"},
    "uncapped_v08": {"results": "hvg_{flavor}_uncapped_v08", "axis": "hvg_{flavor}_a_uncapped_v08.h5ad",
                    "suffix": "_uncapped_v08", "short": "v08"},
    "uncapped_v08_iid": {"results": "hvg_{flavor}_a_uncapped_v08_iid",
                         "axis": "hvg_{flavor}_a_uncapped_v08.h5ad",
                         "suffix": "_uncapped_v08_iid", "short": "v08iid"},
}
ms = MODEL_SETS[MODEL_SET]
FLAV = f"{FLAVOR}{ms['suffix']}"
AXIS_H5AD = os.path.join(D, ms["axis"].format(flavor=FLAVOR))
GENE_TXT = os.path.join(D, f"gene_lists/hvg_{FLAVOR}_a_uncapped_v08_genes.txt")
EVAL_IMPACT_TAG = f"{TAG_CTRL}_impact_{FLAVOR}_{ms['short']}"
EVAL_SCGEN_TAG = f"{TAG_CTRL}_scgen_{FLAVOR}_{ms['short']}"
PLOT_OUT = os.path.join(REPO, "speciesOT/baseline/analysis/paper_style_bcg_outputs",
                        f"{TAG_CTRL}_{FLAVOR}_{MODEL_SET}")
HUMAN_OUT = os.path.join(D, f"bcg_human_unvax_target_{FLAV}.h5ad")
HUMAN_OUT07 = os.path.join(D, f"bcg_human_unvax_target_{FLAV}_anndata07.h5ad")


def _dense(X):
    return X.toarray() if sp_sparse.issparse(X) else np.asarray(X)


def _strip(g):
    g = str(g)
    return g.split(".")[0] if g.startswith("ENS") and "." in g else g


def _run(cmd, cwd=None):
    print("\n$", " ".join(cmd))
    subprocess.run(cmd, cwd=cwd or REPO, check=True)


if os.path.exists(GENE_TXT):
    axis_genes = [g.strip() for g in open(GENE_TXT) if g.strip()]
else:
    _ax = ad.read_h5ad(AXIS_H5AD, backed="r")
    axis_genes = [_strip(g) for g in _ax.var_names.astype(str)]
    _ax.file.close()
assert len(axis_genes) == 1000, len(axis_genes)

# --- 1. verify posed mouse files ---
def verify_posed_mouse(path, label, expect_min_overlap=900):
    print("=" * 72)
    print(label, path)
    if not os.path.exists(path) or str(path).startswith("/path/to/"):
        raise FileNotFoundError(f"set {label} to a real file: {path}")
    a = sc.read_h5ad(path)
    names = [_strip(g) for g in a.var_names.astype(str)]
    n_ensg = sum(g.startswith("ENSG") for g in names)
    layers = list(a.layers.keys()) if a.layers else []
    print(f"  shape {a.shape}  layers {layers}  ENSG {n_ensg}/{a.n_vars}")
    if "cell_type" in a.obs:
        print("  cell_type", a.obs["cell_type"].astype(str).value_counts().to_dict())
    checks = [("human ENSG names", n_ensg >= max(10, int(0.8 * a.n_vars))),
              ("layers has counts", "counts" in a.layers)]
    if "counts" in a.layers:
        C = _dense(a.layers["counts"]).astype(np.float64)
        frac_int = float(np.mean(np.isclose(C, np.round(C), atol=1e-5)))
        print(f"  counts min={C.min():.4g} max={C.max():.4g} mean={C.mean():.4g} frac_int={frac_int:.4f}")
        checks += [("counts continuous (posed)", frac_int < 0.05),
                   ("counts look count-scale, not log1p", float(C.max()) > 15)]
        if "counts_original" in a.layers:
            O = _dense(a.layers["counts_original"]).astype(np.float64)
            lib_rel = np.abs(C.sum(1) - O.sum(1)) / np.maximum(O.sum(1), 1e-6)
            print(f"  counts_original max={O.max():.4g}  lib |posed-orig|/orig median={np.median(lib_rel):.3g}")
            checks += [("counts_original integer", float(np.mean(np.isclose(O, np.round(O), atol=1e-5))) > 0.99),
                       ("posed library ≈ original UMI total", float(np.median(lib_rel)) < 0.05)]
    overlap = len(set(names) & set(axis_genes))
    print(f"  {FLAVOR} {MODEL_SET} overlap {overlap}/{len(axis_genes)}")
    if overlap < 700:
        print("  WARNING: looks like the old v07/685-gene export, not the v08 retrain")
    checks.append((f"axis overlap ≥ {expect_min_overlap}", overlap >= expect_min_overlap))
    ok = True
    for name, passed in checks:
        print(("    PASS  " if passed else "    FAIL  ") + name)
        ok = ok and passed
    if not ok:
        raise RuntimeError(f"{label} failed the posed-file checklist — do not predict")
    print("  READY for --posed-ensg")
    return a


verify_posed_mouse(MOUSE_CTRL, "MOUSE_CTRL")
verify_posed_mouse(MOUSE_TRT, "MOUSE_TRT")

# --- 2. predict both arms ---
for mouse_path, tag in [(MOUSE_CTRL, TAG_CTRL), (MOUSE_TRT, TAG_TRT)]:
    _run(["bash", PREDICT_SH, "--model-set", MODEL_SET, "--posed-ensg",
          mouse_path, tag])

# --- 3. confirm aligned gene axis ---
def aligned(tag, kind):
    names = {
        "aligned": f"{tag}_aligned_{FLAV}.h5ad",
        "aligned07": f"{tag}_aligned_{FLAV}_anndata07.h5ad",
        "impact": f"{tag}_predicted_human_via_impact_cellot_{FLAV}.h5ad",
        "scgen": f"{tag}_predicted_human_via_scgen_{FLAV}.h5ad",
    }
    return os.path.join(D, names[kind])


for tag in (TAG_CTRL, TAG_TRT):
    p = aligned(tag, "aligned")
    a = sc.read_h5ad(p)
    names = [_strip(g) for g in a.var_names.astype(str)]
    X = _dense(a.X)
    print(f"{tag} aligned {a.shape}  order==axis {names == axis_genes}  "
          f".X max={float(X.max()):.3g} (expect log1p, max<15)")
    if names != axis_genes:
        raise RuntimeError(f"{tag} aligned gene order != {FLAVOR} {MODEL_SET} list")

# --- 4. human unvaccinated → model axis + log1p ---
if not os.path.exists(HUMAN_CTRL) or str(HUMAN_CTRL).startswith("/path/to/"):
    raise FileNotFoundError(f"set HUMAN_CTRL: {HUMAN_CTRL}")
src = sc.read_h5ad(HUMAN_CTRL)
if "counts" in src.layers:
    src.X = src.layers["counts"].astype(np.float32)
X = _dense(src.X).astype(np.float32)
if not np.allclose(X[:50].ravel(), np.round(X[:50].ravel())):
    raise ValueError("HUMAN_CTRL must be raw integer counts")
pos = {_strip(g): i for i, g in enumerate(src.var_names.astype(str))}
Xn = np.zeros((src.n_obs, len(axis_genes)), dtype=np.float32)
hit = 0
for j, g in enumerate(axis_genes):
    i = pos.get(_strip(g))
    if i is not None:
        Xn[:, j] = X[:, i]
        hit += 1
print(f"human coverage {hit}/{len(axis_genes)}")
if hit < 900:
    raise RuntimeError("human coverage too low — fix the gene axis before scoring")
h = ad.AnnData(X=Xn, obs=src.obs.copy(), var=pd.DataFrame(index=pd.Index(axis_genes, name="ensg")))
sc.pp.normalize_total(h, target_sum=1e4)
sc.pp.log1p(h)
h.write_h5ad(HUMAN_OUT)

# --- 5. anndata 0.7 rewrite (CellOT env) ---
_run([CELLOT_PY, H5AD_TO_V07, HUMAN_OUT, HUMAN_OUT07])

# --- 6–8. eval IMPACT+scGen, then paper-style figures ---
_run([
    "bash", SCORE_SH,
    "--model-set", MODEL_SET,
    "--flavor", FLAVOR,
    "--tag", TAG_CTRL,
    "--target", HUMAN_OUT07,
])

# --- 9. print scorecards + show boards ---
keep = ["tag", "ncells", "model_over_floor", "r2_model_dec",
        "mmd_model", "mmd_ae_recon_floor", "mmd_decoded_ceiling",
        "frac_gap_closed_decoded", "mean_js"]
rows = []
for tag in (EVAL_IMPACT_TAG, EVAL_SCGEN_TAG):
    p = os.path.join(EVAL_ROOT, tag, "external_target_metrics.csv")
    print(("OK " if os.path.exists(p) else "MISSING ") + p)
    if os.path.exists(p):
        df = pd.read_csv(p)
        df["tag"] = tag
        rows.append(df)
if rows:
    out = pd.concat(rows, ignore_index=True)
    print(out[[c for c in keep if c in out.columns]].to_string(index=False))
print("\nTrust model_over_floor and r2_model_dec. Do not rank BCG on "
      "frac_gap_closed_decoded.")
print(f"figures → {PLOT_OUT}")
for stem in ("scatter_mean_r2", "mmd_bars", "kde_markers", "umap_joint", "dotplot_markers"):
    png = os.path.join(PLOT_OUT, f"{stem}.png")
    if os.path.exists(png):
        display(Image(filename=png, width=720))
print(f"\nDONE. Treated predictions are on disk as {TAG_TRT}_predicted_human_via_* — "
      "score them only against a vaccinated human file "
      f"(same wrapper: --tag {TAG_TRT} --target <vax_human_anndata07>).")
